# 4장 운영 관측 계약 확인

## 이번 질문

이 노트북은 확인 결과를 세 범위로 나눕니다.

- `static_contract`: 설정과 대시보드 JSON이 서로 맞는지 확인한 결과
- `local_observation`: 현재 Risk API의 `/metrics`를 직접 읽은 결과
- `dashboard_observation`: 강사가 제공한 Grafana 화면에서 환경, Scenario와 시간 범위를 선택해 확인할 결과

정적 규약 검사가 통과해도 Grafana가 실제 자료를 수집했다는 뜻은 아닙니다. 정답 없는 운영 신호로 모델 성능을 다시 계산하지도 않습니다. 접속 정보, Alloy와 대시보드는 강사 또는 환경 담당자가 준비하며, 안내받지 못했다면 PREPARED/OFFLINE 경로를 사용합니다.

## 먼저 예상

`baseline`과 `current-shift` 가운데 어느 시나리오에서 고위험 예측 비율과 점수 P95가 높을지 예상합니다. 정답이 없을 때 이 차이로 말할 수 없는 내용도 함께 적습니다.

## 실행과 관측

In [ ]:
from pathlib import Path

import json
import pandas as pd
import yaml

# 1. 저장소 루트를 찾는다.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

pd.DataFrame(
    {
        "경로": [
            "configs/observability/telemetry.yaml",
            "configs/serving/api.yaml",
            "deploy/grafana-cloud/dashboards/ai-quality.json",
        ]
    },
    index=["관측 정책", "API 지표 선언", "대시보드 JSON"],
)


In [ ]:
# 1. 세 파일을 연다.
platform_policy = yaml.safe_load(
    (ROOT / "configs/observability/telemetry.yaml").read_text()
)
api_config = yaml.safe_load((ROOT / "configs/serving/api.yaml").read_text())
dashboard = json.loads(
    (ROOT / "deploy/grafana-cloud/dashboards/ai-quality.json").read_text()
)
assert set(platform_policy) == {"schema_version", "service_namespace", "logging"}

# 2. API가 내보낸다고 선언한 지표 이름.
declared_metrics = set(api_config["observability"]["metrics"].values())

# 3. 대시보드 Prometheus 패널의 조회식을 모은다.
metric_queries = []
for panel in dashboard["panels"]:
    if panel.get("datasource", {}).get("type") != "prometheus":
        continue
    for target in panel.get("targets", []):
        metric_queries.append(target["expr"])
query_text = "\n".join(metric_queries)

# 4. 대시보드 변수 scenario 와 API가 허용한 시나리오가 같은지 본다.
variables = {}
for variable in dashboard["templating"]["list"]:
    variables[variable["name"]] = variable
scenario_values = set(variables["scenario"]["query"].split(","))
allowed_scenarios = set(api_config["observability"]["allowed_scenarios"])
static_contract = {
    "scope": "static_contract",
    "status": "VERIFIED",
    "dashboard_uid": dashboard["uid"],
    "scenario_values": sorted(scenario_values),
}

pd.DataFrame(
    {
        "값": [
            dashboard["uid"],
            len(declared_metrics),
            ", ".join(sorted(scenario_values)),
        ]
    },
    index=["dashboard_uid", "선언 지표 수", "시나리오"],
)


### 1. 대시보드 패널과 데이터 소스 확인

패널마다 Prometheus / Loki / Tempo 중 무엇을 쓰는지 표로 본다.


In [ ]:
# 1. 각 패널의 제목과 데이터 소스 종류를 표로 본다.
# 각 패널의 제목과 데이터 소스 종류.
rows = []
for panel in dashboard["panels"]:
    rows.append(
        {
            "title": panel["title"],
            "type": panel["type"],
            "datasource": panel.get("datasource", {}).get("type", ""),
        }
    )
panels = pd.DataFrame(rows)
panels


### 2. 지표와 조회식 일치 확인

API가 선언한 지표 이름이 대시보드 조회식 문자열 안에 있는지 본다.


In [ ]:
# 1. 선언한 지표가 대시보드 조회식에 쓰이는지 표시한다.
# 선언한 지표가 대시보드 조회식에 실제로 쓰이는지 표시한다.
rows = []
for metric in sorted(declared_metrics):
    rows.append({"metric": metric, "used_by_dashboard": metric in query_text})
metric_usage = pd.DataFrame(rows)
metric_usage


### 3. 로컬 관측과 대상 관측을 나눠 확인

`/metrics`를 읽을 수 있으면 로컬 관측이다. Grafana URL만 있다고 대상 관측이 끝난 것은 아니다.


In [ ]:
import os

import requests

LOCAL_API_URL = "http://127.0.0.1:8000"
dashboard_url = os.getenv("AIQA_GRAFANA_DASHBOARD_URL")
status = None
detail = None
metric_series = []

# 1. 로컬 Compose Risk API /metrics 를 읽는다. 대상 URL과 섞지 않는다.
try:
    metrics_response = requests.get(f"{LOCAL_API_URL}/metrics", timeout=3)
    metrics_response.raise_for_status()
    for line in metrics_response.text.splitlines():
        if line.startswith("aiqa_risk_") and "model_version" in line:
            metric_series.append(line)
except requests.HTTPError as error:
    status = "API_METRICS_UNAVAILABLE"
    detail = str(error)
except requests.RequestException as error:
    status = "API_UNREACHABLE"
    detail = str(error)

# 2. 시계열이 없으면 아직 요청을 안 보낸 것이다. URL은 별도 확인이다.
dashboard_observation = None
if status is None and not metric_series:
    status = "PREDICTION_SERIES_NOT_YET_OBSERVED"
elif status is None and dashboard_url is None:
    status = "LOCAL_METRICS_AVAILABLE"
    dashboard_observation = "DASHBOARD_URL_NOT_CONFIGURED"
elif status is None:
    status = "LOCAL_METRICS_AVAILABLE"
    dashboard_observation = "BROWSER_REVIEW_REQUIRED"

live_telemetry = {
    "scope": "local_observation",
    "status": status,
    "detail": detail,
    "api_url": LOCAL_API_URL,
    "dashboard_url": dashboard_url,
    "dashboard_observation": dashboard_observation,
    "model_series": metric_series[:5],
}
if status == "API_METRICS_UNAVAILABLE":
    live_telemetry["next_action"] = "로컬 Risk API /metrics endpoint를 확인합니다."
elif status == "API_UNREACHABLE":
    live_telemetry["next_action"] = "Compose Risk API를 시작하고 http://127.0.0.1:8000/metrics를 다시 읽습니다."
elif status == "PREDICTION_SERIES_NOT_YET_OBSERVED":
    live_telemetry["next_action"] = "baseline 요청을 보낸 뒤 /metrics를 다시 읽습니다."
elif dashboard_observation == "DASHBOARD_URL_NOT_CONFIGURED":
    live_telemetry["next_action"] = "강사에게 대시보드 URL을 확인하거나 PREPARED/OFFLINE 경로를 사용합니다."

pd.DataFrame(
    [
        {
            "scope": live_telemetry["scope"],
            "status": live_telemetry["status"],
            "dashboard_observation": live_telemetry["dashboard_observation"],
        }
    ]
)


## 해석과 기록

정적 계약, 로컬 관측과 대상 관측을 구분해 누적 기록에 옮깁니다. URL만 있거나 시계열이 아직 없으면 확인 완료로 바꾸지 않습니다.

## 결과 점검

In [ ]:
datasource_types = {
    panel.get("datasource", {}).get("type")
    for panel in dashboard["panels"]
}
assert dashboard["uid"] == "tta-aiqa-quality"
assert all(metric in query_text for metric in declared_metrics)
assert {"prometheus", "loki", "tempo"} <= datasource_types
assert "model_version" in query_text
assert scenario_values == allowed_scenarios
assert all('scenario=~"${scenario:regex}"' in query for query in metric_queries)
print("Static dashboard and telemetry contracts passed.")

## 다음 확인

강사가 제공한 Grafana 대시보드를 연 뒤 다음 순서로 확인합니다.

1. 환경과 Scenario를 선택하고 조회 시간 범위를 고정합니다.
2. `High-risk prediction rate`, 점수, 결측, 상태와 지연을 비교합니다.
3. JSONL에서 run ID, request ID, record ID를 고른 뒤 대시보드 필터 또는 Loki에서 같은 값을 찾습니다.
4. 같은 request ID 또는 trace ID로 Tempo trace의 parent-child 순서를 확인합니다.
5. record ID로 `data/splits-v2/operational.csv`를 잇되, 원본 feature는 Cloud 로그에 올리지 않습니다.
6. 확인한 URL과 시간 범위를 누적 판단 기록에 적습니다.

입력 규약 위반 422는 상태 코드별 요청 건수에서 확인합니다. 5xx 오류율에 422가 없다는 사실을 입력 오류가 없었다고 해석하지 않습니다.